# Chương 2: Sửa lỗi chính tả truyền thống
Pipeline chính:

**Input sai → phát hiện lỗi → sinh ứng viên bằng Edit Distance → xếp hạng bằng N-gram + Noisy Channel → Top-k câu sửa**

### Cách sử dụng notebook
Mỗi phần gồm:
1. **Mẫu**: mã nguồn tham khảo tối thiểu.
2. **Thử nghiệm mẫu**: chạy để hiểu cơ chế.
3. **Task sinh viên**: sinh viên thực hiện cải tiến theo yêu cầu.
4. **Ô TODO**: nơi sinh viên viết phần mở rộng.
5. **Câu hỏi phân tích**: dùng cho báo cáo.



## 0. Chuẩn bị dữ liệu mẫu
Corpus nhỏ bên dưới chỉ phục vụ minh họa. Sinh viên cần thay bằng corpus lớn hơn.

In [ ]:
from collections import Counter, defaultdict
import math
import re
import unicodedata

CORPUS = """
sinh viên học xử lý ngôn ngữ tự nhiên
sinh viên đang học xử lý ngôn ngữ tự nhiên
môn xử lý ngôn ngữ tự nhiên rất thú vị
xử lý ngôn ngữ tự nhiên là một lĩnh vực của trí tuệ nhân tạo
mô hình ngôn ngữ giúp đánh giá xác suất của câu
mô hình bigram sử dụng một từ đứng trước
mô hình trigram sử dụng hai từ đứng trước
khoảng cách chỉnh sửa được dùng để sinh ứng viên
levenshtein distance đo số phép chỉnh sửa tối thiểu
hệ thống sửa lỗi chính tả cần phát hiện lỗi và sửa lỗi
mô hình kênh nhiễu kết hợp mô hình lỗi và mô hình ngôn ngữ
từ điển giúp phát hiện nhiều lỗi non word
lỗi real word cần ngữ cảnh để phát hiện
sinh viên thực hành xây dựng mô hình bigram
sinh viên thực hành xây dựng mô hình trigram
sinh viên viết chương trình sửa lỗi chính tả
sinh viên làm bài tập xử lý văn bản
ngôn ngữ tự nhiên có nhiều hiện tượng nhập nhằng
dữ liệu huấn luyện cần được làm sạch
mô hình học từ dữ liệu văn bản
hôm nay sinh viên học về khoảng cách chỉnh sửa
hôm nay lớp học về mô hình ngôn ngữ
giảng viên hướng dẫn sinh viên làm bài tập
bài tập yêu cầu xây dựng hệ thống sửa lỗi
hệ thống trả về ba câu sửa tốt nhất
chúng ta sử dụng xác suất để xếp hạng ứng viên
tần suất từ có thể hỗ trợ xếp hạng ứng viên
ngữ cảnh giúp chọn từ phù hợp hơn
tôi đang học xử lý ngôn ngữ tự nhiên
tôi đang làm bài tập xử lý ngôn ngữ tự nhiên
tôi thích học xử lý ngôn ngữ tự nhiên
tôi viết chương trình bằng python
python được sử dụng nhiều trong xử lý ngôn ngữ tự nhiên
mô hình truyền thống có thể dùng bigram và trigram
mô hình hiện đại có thể dùng transformer
chính tả tiếng việt có dấu thanh
tiếng việt có nhiều từ ghép
viết sai một ký tự có thể tạo ra từ không tồn tại
một số lỗi tạo ra từ vẫn hợp lệ
mô hình ngôn ngữ hỗ trợ sửa lỗi theo ngữ cảnh
""".strip().splitlines()

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text.lower())
    text = re.sub(r"[^\w\sÀ-ỹ]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str):
    return normalize_text(text).split()

tokenized_corpus = [tokenize(s) for s in CORPUS]

word_freq = Counter()
for sent in tokenized_corpus:
    word_freq.update(sent)

vocab = set(word_freq)

print("Số câu:", len(tokenized_corpus))
print("Vocabulary size:", len(vocab))
print("10 từ phổ biến:", word_freq.most_common(10))

# Task 1 — Levenshtein Distance

## Mẫu
Cài đặt khoảng cách Levenshtein bằng quy hoạch động với 3 thao tác:
- insertion,
- deletion,
- substitution.

In [ ]:
def levenshtein(a: str, b: str, return_matrix=False):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,        # delete
                dp[i][j - 1] + 1,        # insert
                dp[i - 1][j - 1] + cost  # substitute / match
            )

    return (dp[n][m], dp) if return_matrix else dp[n][m]

for a, b in [("học", "hco"), ("sinh", "sihn"), ("ngôn", "ngonn")]:
    print(a, "->", b, "=", levenshtein(a, b))

## Yêu cầu Task 1
**Mức cơ bản**
1. Viết lại Levenshtein bằng **2 hàng bộ nhớ** thay vì toàn bộ ma trận.
2. So sánh kết quả với bản mẫu trên ít nhất 20 cặp từ.

**Mức khá**
3. Cài đặt **Damerau–Levenshtein** để xử lý đảo ký tự, ví dụ `đnag → đang`.
4. Đo thời gian chạy khi độ dài chuỗi tăng dần.

**Mức nâng cao**
5. Cài đặt **weighted edit distance** cho một số cặp dễ nhầm tiếng Việt.
6. Giải thích khi nào weighted distance tốt hơn distance chuẩn.

### TODO

In [ ]:
def levenshtein_optimized(a: str, b: str):
    # TODO: chỉ dùng O(min(len(a), len(b))) bộ nhớ
    pass

def damerau_levenshtein(a: str, b: str):
    # TODO: thêm thao tác transpose hai ký tự kề nhau
    pass

### Câu hỏi phân tích Task 1
- Vì sao Levenshtein có độ phức tạp thời gian \(O(nm)\)?
- Nếu chỉ cần **distance** mà không cần truy vết, có thể giảm bộ nhớ thế nào?
- Trường hợp nào `đnag → đang` cho thấy Levenshtein chuẩn chưa tối ưu?

# Task 2 — Truy vết chuỗi thao tác chỉnh sửa

## Code mẫu
Từ ma trận DP, đi ngược để tìm các thao tác biến từ nguồn thành đích.

In [ ]:
def levenshtein_trace(a: str, b: str):
    dist, dp = levenshtein(a, b, return_matrix=True)
    i, j = len(a), len(b)
    ops = []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1] and dp[i][j] == dp[i-1][j-1]:
            ops.append(("MATCH", a[i-1], b[j-1]))
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(("SUBSTITUTE", a[i-1], b[j-1]))
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(("DELETE", a[i-1], ""))
            i -= 1
        else:
            ops.append(("INSERT", "", b[j-1]))
            j -= 1

    ops.reverse()
    return dist, ops

d, ops = levenshtein_trace("sinh", "sihn")
print("Distance:", d)
for op in ops:
    print(op)

## Yêu cầu Task 2
1. Viết hàm in truy vết dưới dạng dễ đọc, ví dụ:
   `SUBSTITUTE: h → c`.
2. Đếm riêng số thao tác `I, D, S`.
3. Mở rộng để truy vết cả **TRANSPOSE** trong Damerau–Levenshtein.
4. Trực quan hóa ma trận DP bằng bảng hoặc heatmap.
5. Chọn 10 lỗi tiếng Việt và phân tích loại thao tác thường gặp nhất.

### TODO

In [ ]:
def pretty_trace(a: str, b: str):
    # TODO: trả về chuỗi mô tả các thao tác chỉnh sửa
    pass

def count_edit_operations(ops):
    # TODO: trả về dictionary {'insert': ..., 'delete': ..., 'substitute': ...}
    pass

# Task 3 — Xây dựng Bigram / Trigram Language Model

## Code mẫu
Mô hình N-gram dùng giả định Markov:

Bigram:
\[
P(w_i|w_{i-1})
\]

Trigram:
\[
P(w_i|w_{i-2}, w_{i-1})
\]

Bản mẫu dùng **Add-k smoothing**.

In [ ]:
class NgramLanguageModel:
    def __init__(self, n=2, k=0.1):
        assert n in (1, 2, 3)
        self.n = n
        self.k = k
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()

    def fit(self, tokenized_sentences):
        for sent in tokenized_sentences:
            tokens = ["<s>"] * (self.n - 1) + sent + ["</s>"]
            self.vocab.update(sent)
            self.vocab.add("</s>")

            for i in range(self.n - 1, len(tokens)):
                ngram = tuple(tokens[i-self.n+1:i+1]) if self.n > 1 else (tokens[i],)
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
        return self

    def prob(self, word, context=()):
        if self.n == 1:
            context = ()
        else:
            context = tuple(context[-(self.n-1):])
            if len(context) < self.n - 1:
                context = ("<s>",) * (self.n - 1 - len(context)) + context

        ngram = context + (word,)
        V = len(self.vocab) + 1
        return (self.ngram_counts[ngram] + self.k) / (
            self.context_counts[context] + self.k * V
        )

    def sentence_logprob(self, tokens):
        padded = ["<s>"] * (self.n - 1) + tokens + ["</s>"]
        score = 0.0
        for i in range(self.n - 1, len(padded)):
            word = padded[i]
            context = padded[max(0, i-self.n+1):i]
            score += math.log(self.prob(word, context) + 1e-15)
        return score

bigram_lm = NgramLanguageModel(n=2, k=0.1).fit(tokenized_corpus)
trigram_lm = NgramLanguageModel(n=3, k=0.1).fit(tokenized_corpus)

s = tokenize("sinh viên đang học xử lý ngôn ngữ tự nhiên")
print("Bigram :", bigram_lm.sentence_logprob(s))
print("Trigram:", trigram_lm.sentence_logprob(s))

## Yêu cầu Task 3
1. Thêm **Unigram LM** và so sánh với Bigram/Trigram.
2. Thử các giá trị `k = 1.0, 0.5, 0.1, 0.01`.
3. Cài **Interpolation** giữa unigram, bigram và trigram.
4. Cài một phiên bản **Backoff** đơn giản.
5. Tính **perplexity** trên tập validation.
6. Tìm trường hợp trigram tệ hơn bigram và giải thích do data sparsity.

### TODO

In [ ]:
def perplexity(lm, tokenized_sentences):
    # TODO: tính perplexity trung bình trên corpus
    pass

def interpolated_prob(word, context, unigram_lm, bigram_lm, trigram_lm,
                      l1=0.1, l2=0.3, l3=0.6):
    # TODO: nội suy xác suất từ ba mô hình
    pass

# Task 4 — Sinh ứng viên sửa lỗi

## Code mẫu
Duyệt vocabulary, giữ các từ có edit distance không vượt quá bán kính cho trước.

In [ ]:
def generate_candidates(observed,
                        vocabulary,
                        freq,
                        max_distance=2,
                        top_n=20):
    scored = []
    for cand in vocabulary:
        d = levenshtein(observed, cand)
        if d <= max_distance:
            scored.append((cand, d, freq[cand]))

    scored.sort(key=lambda x: (x[1], -x[2], x[0]))
    return scored[:top_n]

for typo in ["hco", "ngonn", "sihn"]:
    print("\n", typo, "=>", generate_candidates(typo, vocab, word_freq, 2, 10))

## Yêu cầu Task 4
1. Thêm điều kiện lọc theo **độ dài từ** để giảm số candidate.
2. Thử `max_distance = 1, 2, 3` và đo:
   - recall của từ đúng,
   - số candidate trung bình,
   - thời gian.
3. Cài **BK-tree** hoặc một cấu trúc tìm kiếm gần đúng khác.
4. So sánh brute force với BK-tree trên vocabulary lớn hơn.
5. Tạo **confusion set** riêng cho:
   - `s/x`,
   - `ch/tr`,
   - `d/gi/r`,
   - thiếu dấu thanh,
   - lỗi Telex/VNI.
6. Đề xuất cách xử lý viết liền/viết tách.

### TODO

In [ ]:
def generate_candidates_fast(observed, vocabulary, freq,
                             max_distance=2, top_n=20):
    # TODO: cải tiến hiệu năng candidate generation
    # Gợi ý: lọc theo |len(cand)-len(observed)| <= max_distance
    pass

# Task 5 — Error Model \(P(x|w)\)

## Code mẫu
Phiên bản đơn giản sử dụng:

\[
P(x|w) \propto \exp(-\alpha d(x,w))
\]

Distance càng lớn → xác suất lỗi càng nhỏ.

In [ ]:
def error_logprob(observed, candidate, alpha=1.7):
    d = levenshtein(candidate, observed)
    return -alpha * d

for cand in ["học", "họ", "hôm"]:
    print(cand, error_logprob("hco", cand))

## Yêu cầu Task 5
1. Tạo tập dữ liệu cặp `(correct_word, typo)`.
2. Thống kê confusion matrix cho:
   - insert,
   - delete,
   - substitute,
   - transpose.
3. Ước lượng xác suất lỗi từ dữ liệu thay vì chỉ dùng `exp(-αd)`.
4. Cho chi phí thấp hơn với các phím gần nhau trên bàn phím.
5. Thử một error model riêng cho lỗi dấu tiếng Việt.
6. So sánh accuracy trước và sau khi dùng confusion-based error model.

### TODO

In [ ]:
def train_confusion_model(error_pairs):
    """
    error_pairs: list[(correct_word, observed_typo)]
    TODO: thống kê các loại lỗi và ước lượng xác suất.
    """
    pass

def confusion_error_logprob(observed, candidate, confusion_model):
    # TODO: tính log P(observed | candidate)
    pass

# Task 6 — Noisy Channel Ranking

## Code mẫu

\[
\hat{w} = \arg\max_w P(w|x)
\propto P(x|w)P(w)
\]

Trong ngữ cảnh:

\[
score(w)
=
\log P(x|w)
+
\lambda\log P(w|context)
\]

In [ ]:
def candidate_score(observed,
                    candidate,
                    left_context,
                    lm,
                    word_freq,
                    alpha=1.7,
                    lm_weight=1.0,
                    freq_weight=0.15):
    err = error_logprob(observed, candidate, alpha=alpha)
    lm_log = math.log(lm.prob(candidate, left_context) + 1e-15)
    freq_log = math.log(word_freq[candidate] + 1)

    total = err + lm_weight * lm_log + freq_weight * freq_log
    return total

def rank_word_candidates(sentence, error_index, lm, max_distance=2, top_k=5):
    tokens = tokenize(sentence)
    observed = tokens[error_index]

    cands = generate_candidates(
        observed, vocab, word_freq,
        max_distance=max_distance, top_n=50
    )

    rows = []
    for cand, dist, freq in cands:
        left_context = tokens[max(0, error_index-(lm.n-1)):error_index]
        score = candidate_score(
            observed, cand, left_context, lm, word_freq
        )
        rows.append((cand, dist, freq, score))

    rows.sort(key=lambda x: x[-1], reverse=True)
    return rows[:top_k]

sentence = "sinh viên đang hco xử lý ngôn ngữ tự nhiên"
rank_word_candidates(sentence, 3, trigram_lm)

## Yêu cầu Task 6
1. Tuning `alpha`, `lm_weight`, `freq_weight` trên validation set.
2. Dùng **cả left context và right context** thay vì chỉ dùng từ trước.
3. So sánh:
   - chỉ Edit Distance,
   - Edit Distance + frequency,
   - Edit Distance + Bigram,
   - Edit Distance + Trigram,
   - Noisy Channel đầy đủ.
4. Trả về **confidence score** cho mỗi đề xuất.
5. Đề xuất cơ chế không sửa nếu score cải thiện quá ít.
6. Phân tích tối thiểu 10 trường hợp ranking sai.

### TODO

In [ ]:
def bidirectional_candidate_score(observed, candidate,
                                  left_context, right_context,
                                  lm, word_freq):
    # TODO: sử dụng cả ngữ cảnh bên trái và bên phải
    pass

# Task 7 — Sửa nhiều lỗi bằng Beam Search

## Code mẫu
Beam Search giữ lại một số hypothesis tốt nhất sau mỗi token.

In [ ]:
def correction_candidates_for_token(token, max_distance=2, max_candidates=8):
    if token in vocab:
        return [(token, 0)]

    cands = generate_candidates(
        token, vocab, word_freq,
        max_distance=max_distance,
        top_n=max_candidates
    )
    return [(w, d) for w, d, _ in cands] or [(token, 99)]

def correct_sentence(sentence,
                     lm,
                     beam_width=20,
                     top_k=3,
                     max_distance=2,
                     max_candidates=8,
                     alpha=1.7,
                     lm_weight=1.0):

    observed_tokens = tokenize(sentence)
    beam = [([], 0.0)]

    for observed in observed_tokens:
        candidates = correction_candidates_for_token(
            observed,
            max_distance=max_distance,
            max_candidates=max_candidates
        )

        new_beam = []

        for prefix, prefix_score in beam:
            for cand, _dist in candidates:
                left_context = prefix[-(lm.n-1):] if lm.n > 1 else []

                err = error_logprob(observed, cand, alpha=alpha)
                lm_log = math.log(lm.prob(cand, left_context) + 1e-15)
                freq_log = 0.10 * math.log(word_freq[cand] + 1)

                score = prefix_score + err + lm_weight * lm_log + freq_log
                new_beam.append((prefix + [cand], score))

        new_beam.sort(key=lambda x: x[1], reverse=True)
        beam = new_beam[:beam_width]

    final = []
    for toks, score in beam:
        end_context = toks[-(lm.n-1):] if lm.n > 1 else []
        score += math.log(lm.prob("</s>", end_context) + 1e-15)
        final.append((" ".join(toks), score))

    final.sort(key=lambda x: x[1], reverse=True)
    return final[:top_k]

for rank, item in enumerate(
    correct_sentence(
        "sinh viên đang hco xử lý ngôn ngữ tự nhiên",
        trigram_lm,
        top_k=3
    ), 1
):
    print(rank, item)

## Yêu cầu Task 7
1. Thử `beam_width = 1, 5, 10, 20, 50`.
2. Đo trade-off giữa:
   - thời gian,
   - Top-1 accuracy,
   - Top-3 accuracy.
3. Thêm penalty để tránh sửa quá nhiều từ đúng.
4. Cho phép giữ nguyên non-word nếu không có candidate đủ tốt.
5. Mở rộng xử lý đồng thời **non-word + real-word error**.
6. So sánh Beam Search với Greedy Search.

### TODO

In [ ]:
def correct_sentence_improved(sentence, lm, beam_width=20, top_k=3):
    # TODO: thêm penalty sửa từ, confidence threshold,
    # và/hoặc xử lý real-word error.
    pass

# Task 8 — Real-word Error

## Code mẫu
Real-word error không thể phát hiện chỉ bằng từ điển.

Ví dụ:
- `"làm"` là từ hợp lệ.
- Nhưng trong một số ngữ cảnh `"sinh viên đang làm xử lý..."`, từ `"học"` có thể phù hợp hơn.

Ta dùng một **confusion set** nhỏ để minh họa.

In [ ]:
CONFUSION_SETS = {
    "làm": ["làm", "học"],
    "học": ["học", "làm"],
}

def detect_and_correct_realword(sentence, lm, threshold=0.2):
    tokens = tokenize(sentence)
    suggestions = []

    for i, observed in enumerate(tokens):
        if observed not in CONFUSION_SETS:
            continue

        left_context = tokens[max(0, i-(lm.n-1)):i]
        scored = []

        for cand in CONFUSION_SETS[observed]:
            lp = math.log(lm.prob(cand, left_context) + 1e-15)
            scored.append((cand, lp))

        scored.sort(key=lambda x: x[1], reverse=True)

        current_score = next(v for w, v in scored if w == observed)
        best_word, best_score = scored[0]

        if best_word != observed and best_score - current_score > threshold:
            suggestions.append(
                (i, observed, best_word, best_score-current_score)
            )

    return suggestions

print(
    detect_and_correct_realword(
        "sinh viên đang làm xử lý ngôn ngữ tự nhiên",
        trigram_lm
    )
)

## Yêu cầu Task 8
1. Tự xây dựng ít nhất **10 confusion sets** tiếng Việt.
2. Dùng trigram hoặc bidirectional context để đánh giá.
3. Tuning `threshold` trên validation set.
4. Đo:
   - precision phát hiện lỗi,
   - recall phát hiện lỗi,
   - F1.
5. Phân tích false positive: hệ thống sửa một từ vốn đã đúng.
6. Đề xuất cách chuyển từ confusion-set thủ công sang mô hình học từ dữ liệu.

### TODO

In [ ]:
MY_CONFUSION_SETS = {
    # TODO: bổ sung ít nhất 10 confusion sets
}

def realword_detector_improved(sentence, lm, confusion_sets, threshold=0.2):
    # TODO
    pass

# Task 9 — Đánh giá hệ thống

## Code mẫu: CER, WER và Top-k Accuracy

In [ ]:
def edit_distance_sequence(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1,n+1):
        for j in range(1,m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )
    return dp[n][m]

def cer(reference, hypothesis):
    ref = list(normalize_text(reference).replace(" ", ""))
    hyp = list(normalize_text(hypothesis).replace(" ", ""))
    return edit_distance_sequence(ref, hyp) / max(1, len(ref))

def wer(reference, hypothesis):
    ref = tokenize(reference)
    hyp = tokenize(hypothesis)
    return edit_distance_sequence(ref, hyp) / max(1, len(ref))

ref = "sinh viên đang học xử lý ngôn ngữ tự nhiên"
hyp = "sinh viên đang hco xử lý ngôn ngữ tự nhiên"

print("CER =", cer(ref, hyp))
print("WER =", wer(ref, hyp))

## Yêu cầu Task 9
Tạo tập test tối thiểu **100 câu**, gồm nhiều loại lỗi.

Báo cáo:
1. **Top-1 accuracy**.
2. **Top-3 accuracy**.
3. **CER** trước và sau sửa.
4. **WER** trước và sau sửa.
5. Precision / Recall / F1 cho **phát hiện lỗi**.
6. Kết quả riêng theo:
   - lỗi insert,
   - delete,
   - substitute,
   - transpose,
   - thiếu dấu,
   - real-word.
7. Thời gian xử lý trung bình mỗi câu.

### TODO

In [ ]:
TEST_SET = [
    # ("câu sai", "câu đúng"),
]

def evaluate_system(test_set, correction_fn):
    # TODO: tổng hợp các metric
    pass

# Task 10 — Error Analysis

Đây là phần **phân tích bắt buộc*: 

Sinh viên chọn tối thiểu **20 trường hợp sửa sai** và gán mỗi trường hợp vào một hoặc nhiều nhóm:

| Nhóm lỗi hệ thống | Ví dụ nguyên nhân |
|---|---|
| Detection error | Không phát hiện được real-word |
| Candidate error | Từ đúng không nằm trong candidate set |
| Ranking error | Từ đúng có candidate nhưng bị xếp hạng thấp |
| LM sparsity | N-gram chưa từng thấy context |
| Corpus bias | Corpus lệch miền dữ liệu |
| Error-model mismatch | Mô hình lỗi không phản ánh lỗi thật |
| Tokenization issue | Viết liền/tách hoặc tách từ sai |
| Vietnamese-specific | Dấu thanh, Telex/VNI, phương ngữ |

## Yêu cầu Task 10
Tạo bảng ít nhất 20 dòng gồm:
- input sai,
- gold,
- prediction,
- loại lỗi hệ thống,
- nguyên nhân,
- đề xuất cải tiến.

### TODO

In [ ]:
error_analysis = [
    # {
    #   "input": "...",
    #   "gold": "...",
    #   "prediction": "...",
    #   "category": "ranking error",
    #   "reason": "...",
    #   "proposed_fix": "..."
    # }
]

error_analysis[:3]

# Task 11 — So sánh hệ thống baseline và hệ thống cải tiến

Sinh viên phải báo cáo ít nhất các hệ thống:

### Baseline A
**Edit Distance only**

### Baseline B
**Edit Distance + Frequency**

### Baseline C
**Edit Distance + Bigram**

### Baseline D
**Edit Distance + Trigram + Noisy Channel**

### Hệ thống cải tiến của nhóm
Phải có ít nhất **2 cải tiến** so với baseline D, ví dụ:
- Damerau–Levenshtein,
- weighted edit distance,
- confusion model học từ dữ liệu,
- BK-tree,
- interpolation/backoff,
- bidirectional context,
- real-word detection,
- confidence threshold,
- xử lý lỗi dấu tiếng Việt.

Bảng báo cáo đề xuất:

| Hệ thống | Top-1 | Top-3 | CER | WER | ms/câu |
|---|---:|---:|---:|---:|---:|
| Edit only | | | | | |
| + Frequency | | | | | |
| + Bigram | | | | | |
| + Trigram + Noisy Channel | | | | | |
| **Improved system** | | | | | |

# Bài nộp:

Notebook cuối cùng phải có:

- [ ] Task 1: Levenshtein + ít nhất 1 cải tiến.
- [ ] Task 2: Truy vết thao tác.
- [ ] Task 3: Bigram/Trigram + smoothing.
- [ ] Task 4: Candidate generation.
- [ ] Task 5: Error model.
- [ ] Task 6: Noisy Channel ranking.
- [ ] Task 7: Top-3 correction bằng Beam Search.
- [ ] Task 8: Real-word error.
- [ ] Task 9: Evaluation.
- [ ] Task 10: 20 failure cases.
- [ ] Task 11: So sánh baseline và hệ thống cải tiến.

## Yêu cầu tối thiểu để đạt
- Notebook chạy được từ đầu đến cuối.
- Không dùng thư viện sửa chính tả có sẵn để thay thế phần cốt lõi.
- Có ít nhất **2 cải tiến thực sự** so với hệ thống mẫu.
- Có số liệu chứng minh cải tiến tốt hơn hoặc giải thích vì sao không tốt hơn.
- Có phân tích lỗi, không chỉ báo cáo Accuracy.

## Điểm cộng
- BK-tree / SymSpell-style indexing.
- Error model học từ corpus lỗi thật.
- Xử lý lỗi Telex/VNI.
- Character N-gram.
- Interpolation / backoff.
- So sánh với BARTpho / Transformer.

---
# Rubric

| Thành phần | Điểm |
|---|---:|
| Levenshtein + truy vết | 15 |
| N-gram LM | 15 |
| Candidate generation | 10 |
| Noisy Channel | 15 |
| Real-word / Vietnamese error | 10 |
| Evaluation | 15 |
| Cải tiến hệ thống | 10 |
| Error analysis + trình bày | 10 |
| **Tổng** | **100** |

> Lưu ý: Sinh viên nên ưu tiên **phân tích và lý giải được kết quả thực nghiệm**